# 04. 위험도 모델링

**프로젝트**: 창원시 폭우 침수·하수 역류 우선 대응지역 분석  
**목적**: 격자별 통합 위험도를 산출하고, 우선 대응 지역 TOP 20을 선정

---

### 모델링 구조

```
Layer 1. 침수 취약도 = f(강수량, 배수용량, 지형 저지대)
Layer 2. 하수역류 위험도 = f(관로 노후도, 관경, 민원 이력)
Layer 3. 취약계층 노출도 = f(고령 비율, 1인가구, 반지하)
→ 통합 위험도 = w1·L1 + w2·L2 + w3·L3
```

## 1. 환경 설정

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

PROJECT_ROOT = Path.cwd().parent
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
MODEL_DIR = PROJECT_ROOT / 'models'

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print('모델링 환경 준비 완료')

## 2. 데이터 로드

In [ ]:
# TODO: 전처리 완료 데이터 로드
# grid = gpd.read_file(PROCESSED_DIR / 'analysis_grid.geojson')
# print(f'분석 격자: {len(grid)}개')
# print(f'컬럼: {grid.columns.tolist()}')

print('전처리 데이터 생성 후 로드')

## 3. Layer 1 — 침수 취약도

In [ ]:
def calc_flood_vulnerability(grid):
    """
    침수 취약도 산출
    - 배수펌프 용량 부족 → 위험 ↑
    - 강수량 상위 지역 → 위험 ↑
    - 저지대 (해발고도 낮음) → 위험 ↑
    """
    scaler = MinMaxScaler()
    
    features = ['펌프용량합계', '평균강수량', '평균해발고도']
    
    # 펌프 용량: 낮을수록 위험 → 역수
    grid['pump_risk'] = 1 - scaler.fit_transform(
        grid[['펌프용량합계']].fillna(0)
    )
    
    # 강수량: 높을수록 위험
    grid['rain_risk'] = scaler.fit_transform(
        grid[['평균강수량']].fillna(0)
    )
    
    # 해발고도: 낮을수록 위험 → 역수
    grid['elev_risk'] = 1 - scaler.fit_transform(
        grid[['평균해발고도']].fillna(grid['평균해발고도'].median())
    )
    
    # 침수 취약도 (가중합)
    grid['flood_score'] = (
        0.4 * grid['pump_risk'] +
        0.35 * grid['rain_risk'] +
        0.25 * grid['elev_risk']
    )
    
    return grid

# TODO: 적용
# grid = calc_flood_vulnerability(grid)
# print(grid['flood_score'].describe())
print('침수 취약도 함수 준비 완료')

## 4. Layer 2 — 하수도 역류 위험도

In [ ]:
def calc_sewer_risk(grid):
    """
    하수도 역류 위험도 산출
    - 관로 노후 비율 높을수록 위험
    - 관경 작을수록 위험
    - 민원 이력 많을수록 위험
    """
    scaler = MinMaxScaler()
    
    # 노후 관로 비율: 높을수록 위험
    grid['aging_risk'] = scaler.fit_transform(
        grid[['노후관로비율']].fillna(0)
    )
    
    # 평균 관경: 작을수록 위험 → 역수
    grid['diameter_risk'] = 1 - scaler.fit_transform(
        grid[['평균관경']].fillna(grid['평균관경'].median())
    )
    
    # 민원 건수: 많을수록 위험 (있는 경우만)
    if '민원건수' in grid.columns:
        grid['complaint_risk'] = scaler.fit_transform(
            grid[['민원건수']].fillna(0)
        )
        grid['sewer_score'] = (
            0.4 * grid['aging_risk'] +
            0.3 * grid['diameter_risk'] +
            0.3 * grid['complaint_risk']
        )
    else:
        grid['sewer_score'] = (
            0.55 * grid['aging_risk'] +
            0.45 * grid['diameter_risk']
        )
    
    return grid

# TODO: 적용
# grid = calc_sewer_risk(grid)
# print(grid['sewer_score'].describe())
print('하수도 역류 위험도 함수 준비 완료')

## 5. Layer 3 — 취약계층 노출도

In [ ]:
def calc_vulnerability_exposure(grid):
    """
    취약계층 노출도 산출
    - 고령 인구 비율
    - 1인가구 비율
    - 노후 건물(반지하 포함) 비율
    """
    scaler = MinMaxScaler()
    
    # 고령 비율
    grid['elderly_risk'] = scaler.fit_transform(
        grid[['고령비율']].fillna(0)
    )
    
    # 1인가구 비율
    grid['single_risk'] = scaler.fit_transform(
        grid[['일인가구비율']].fillna(0)
    )
    
    # 노후 건물 비율
    grid['old_bld_risk'] = scaler.fit_transform(
        grid[['노후건물비율']].fillna(0)
    )
    
    grid['vuln_score'] = (
        0.4 * grid['elderly_risk'] +
        0.3 * grid['single_risk'] +
        0.3 * grid['old_bld_risk']
    )
    
    return grid

# TODO: 적용
# grid = calc_vulnerability_exposure(grid)
# print(grid['vuln_score'].describe())
print('취약계층 노출도 함수 준비 완료')

## 6. 통합 위험도 산출 및 TOP 20 선정

In [ ]:
def calc_integrated_risk(grid, w1=0.4, w2=0.35, w3=0.25):
    """
    통합 위험도 = w1·침수취약도 + w2·역류위험도 + w3·취약계층노출도
    가중치 합 = 1.0
    """
    grid['risk_score'] = (
        w1 * grid['flood_score'] +
        w2 * grid['sewer_score'] +
        w3 * grid['vuln_score']
    )
    
    # 등급 분류 (5단계)
    grid['risk_grade'] = pd.cut(
        grid['risk_score'],
        bins=[0, 0.2, 0.4, 0.6, 0.8, 1.0],
        labels=['안전', '관심', '주의', '경계', '위험']
    )
    
    return grid

# TODO: 통합 위험도 산출
# grid = calc_integrated_risk(grid)
# 
# # TOP 20 위험 격자
# top20 = grid.nlargest(20, 'risk_score')[[
#     'grid_id', 'flood_score', 'sewer_score', 'vuln_score',
#     'risk_score', 'risk_grade'
# ]]
# print('=== 우선 대응 지역 TOP 20 ===')
# display(top20)

print('통합 위험도 함수 준비 완료')

## 7. XGBoost 검증 모델 (선택)

민원 데이터가 확보된 경우, 실제 민원 발생 여부를 타겟으로 XGBoost 분류 모델을 학습하여 가중치 기반 점수의 타당성을 검증합니다.

In [ ]:
def train_xgboost_model(grid, target_col='민원발생여부'):
    """
    XGBoost로 민원 발생 예측 → 가중치 검증
    """
    feature_cols = [
        '노후관로비율', '관로총연장', '펌프용량합계',
        '평균강수량', '노후건물비율', '고령비율', '일인가구비율'
    ]
    available = [c for c in feature_cols if c in grid.columns]
    
    X = grid[available].fillna(0)
    y = grid[target_col]
    
    model = xgb.XGBClassifier(
        n_estimators=100,
        max_depth=5,
        learning_rate=0.1,
        random_state=RANDOM_SEED,
        use_label_encoder=False,
        eval_metric='logloss'
    )
    
    scores = cross_val_score(model, X, y, cv=5, scoring='f1')
    print(f'5-Fold CV F1 Score: {scores.mean():.3f} (±{scores.std():.3f})')
    
    # 전체 학습 후 feature importance
    model.fit(X, y)
    importance = pd.Series(
        model.feature_importances_, index=available
    ).sort_values(ascending=True)
    
    fig, ax = plt.subplots(figsize=(8, 5))
    importance.plot(kind='barh', ax=ax, color='#2c3e50')
    ax.set_title('Feature Importance (XGBoost)')
    ax.set_xlabel('Importance')
    plt.tight_layout()
    plt.show()
    
    return model, importance

# TODO: 민원 데이터 확보 시 실행
# model, importance = train_xgboost_model(grid)
print('XGBoost 검증 모델 함수 준비 완료')

## 8. 가중치 민감도 분석

In [ ]:
def sensitivity_analysis(grid):
    """
    가중치 조합별 TOP 20 변화를 분석하여 결과의 안정성을 검증
    """
    weight_sets = [
        (0.4, 0.35, 0.25, '기본'),
        (0.5, 0.3, 0.2, '침수 강조'),
        (0.3, 0.5, 0.2, '역류 강조'),
        (0.3, 0.3, 0.4, '취약계층 강조'),
        (0.33, 0.34, 0.33, '균등'),
    ]
    
    results = {}
    for w1, w2, w3, label in weight_sets:
        score = w1 * grid['flood_score'] + w2 * grid['sewer_score'] + w3 * grid['vuln_score']
        top20_ids = set(grid.nlargest(20, 'risk_score')['grid_id'] if 'risk_score' in grid.columns
                       else score.nlargest(20).index)
        results[label] = top20_ids
    
    # 겹침 비율 산출
    base = results['기본']
    print('=== 가중치 민감도 분석 ===')
    for label, top20 in results.items():
        overlap = len(base & top20) / 20 * 100
        print(f'  {label:15s}: 기본 대비 겹침 {overlap:.0f}%')

# TODO: 적용
# sensitivity_analysis(grid)
print('민감도 분석 함수 준비 완료')

## 9. 모델 및 결과 저장

In [ ]:
# TODO: 결과 저장

# # 통합 위험도 격자 저장
# grid.to_file(PROCESSED_DIR / 'risk_grid.geojson', driver='GeoJSON')
# 
# # TOP 20 별도 저장
# top20.to_csv(PROCESSED_DIR / 'top20_priority_areas.csv', index=False)
# 
# # XGBoost 모델 저장 (있는 경우)
# # import joblib
# # joblib.dump(model, MODEL_DIR / 'xgboost_complaint_predictor.pkl')
# 
# print('=== 모델링 결과 저장 완료 ===')
# print(f'  위험도 격자: {PROCESSED_DIR / "risk_grid.geojson"}')
# print(f'  TOP 20: {PROCESSED_DIR / "top20_priority_areas.csv"}')

print('모델링 완료 후 주석 해제하여 저장')